In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df=pd.read_csv('cardekho_imputated.csv')

In [3]:
df.drop(columns='Unnamed: 0',inplace=True)

In [4]:
df.drop(columns=['car_name','brand'],inplace=True)

In [5]:
df['model']=df['model'].str.strip().str.lower()
df['model'].unique()
df['model']=df['model'].replace('c','c-class')
df['model']=df['model'].replace('redi-go','redigo')
df['model']=df['model'].replace('kuv100','kuv')
df['model'].value_counts().sort_index()

model
3         152
5         118
6          18
7          37
a4         99
         ... 
xl6         7
xuv300     15
xuv500    330
yaris      17
z4          6
Name: count, Length: 117, dtype: int64

In [6]:
model_counts=df['model'].value_counts()
rare=model_counts[model_counts<3].index
rare

Index(['macan', 'nx', 'dzire lxi', 'wrangler', 'rx', 'aura', 'ghibli',
       'gtc4lusso', 'altroz', 'ghost', 'quattroporte', 'gurkha'],
      dtype='object', name='model')

In [7]:
df['model']=df['model'].replace(rare,'other')

In [8]:
num_features=[feature for feature in df.columns if df[feature].dtype!='O']
cat_features=[feature for feature in df.columns if df[feature].dtype=='O']

In [9]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=['selling_price'],axis=1)
y=df['selling_price']

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=1)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder

In [12]:
num_features=X.select_dtypes(exclude='object').columns
ord_features=['model']
ohe_features=['seller_type','fuel_type','transmission_type']

In [13]:
preprocessor=ColumnTransformer(
    transformers=[('ohe',OneHotEncoder(drop='first'),ohe_features),('ord',OrdinalEncoder(),ord_features)],
    remainder='passthrough'
)

In [14]:
X_train=preprocessor.fit_transform(X_train)

In [15]:
X_test=preprocessor.transform(X_test)

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [17]:
def evalutaion(y_test,y_pred):
    mae = mean_absolute_error(y_test,y_pred)
    mse = mean_squared_error(y_test,y_pred)
    rmse = root_mean_squared_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)
    return mae,mse,rmse,r2

In [24]:
models = {
    'Linear R': LinearRegression(),
    'Lasso R' : Lasso(),
    'Ridge R' : Ridge(),
    'KN Reg' : KNeighborsRegressor(),
    'DT R' : DecisionTreeRegressor(),
    'RF R' : RandomForestRegressor(),
    'Ada R': AdaBoostRegressor(),
    'Ada R_hyperparametet': AdaBoostRegressor(n_estimators= 80, loss= 'linear'),
    'SVR' : SVR()
}

In [25]:
for name,model in models.items():
    model.fit(X_train,y_train)
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    print(f"TRAINING EVALUATION : {name}")
    mae,mse,rmse,r2=evalutaion(y_train,y_train_pred)
    print(f"mae : {mae}, mse : {mse}, rmse : {rmse}, r2 : {r2}")

    print(f"TESTING EVALUATION : {name}")
    mae,mse,rmse,r2=evalutaion(y_test,y_test_pred)
    print(f"mae : {mae}, mse : {mse}, rmse : {rmse}, r2 : {r2}")


TRAINING EVALUATION : Linear R
mae : 273847.6620787718, mse : 332188518262.3346, rmse : 576357.9775298808, r2 : 0.6091271908881404
TESTING EVALUATION : Linear R
mae : 265248.77892271796, mse : 209057672231.4792, rmse : 457228.24959912436, r2 : 0.6933214969905077
TRAINING EVALUATION : Lasso R
mae : 273846.1632341757, mse : 332188521746.89844, rmse : 576357.9805527971, r2 : 0.6091271867879948
TESTING EVALUATION : Lasso R
mae : 265247.4404053863, mse : 209057596079.75864, rmse : 457228.166323728, r2 : 0.6933216087017662
TRAINING EVALUATION : Ridge R
mae : 273827.79030213994, mse : 332189288699.47754, rmse : 576358.645896353, r2 : 0.6091262843458798
TESTING EVALUATION : Ridge R
mae : 265240.6863887502, mse : 209061020207.99304, rmse : 457231.91074988747, r2 : 0.6933165856547294
TRAINING EVALUATION : KN Reg
mae : 198257.8844905905, mse : 242565853600.86215, rmse : 492509.7497520858, r2 : 0.7145825596635758
TESTING EVALUATION : KN Reg
mae : 250593.2580017301, mse : 270799503302.4708, rmse : 

In [26]:
## hyper parameter tuning 

params={ 
    'n_estimators': [50,60,70,80,90],
    'loss' : ['linear', 'square', 'exponential'] 
}

In [21]:
from sklearn.model_selection import RandomizedSearchCV
rd=RandomizedSearchCV(estimator=AdaBoostRegressor(),param_distributions=params,n_iter=50,cv=3,verbose=3,n_jobs=-1)

In [22]:
rd.fit(X_train,y_train)

C:\Users\ayush\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 15 is smaller than n_iter=50. Running 15 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 15 candidates, totalling 45 fits


,estimator,AdaBoostRegressor()
,param_distributions,"{'loss': ['linear', 'square', ...], 'n_estimators': [50, 60, ...]}"
,n_iter,50
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [23]:
rd.best_params_

{'n_estimators': 80, 'loss': 'linear'}